In [ ]:
# If needed:
# pip install qiskit qiskit-algorithms qiskit-optimization qiskit-aer

import numpy as np

from qiskit.primitives import StatevectorSampler
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA

from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer


# -----------------------------
# 1. Build a 10-variable binary optimization problem
# -----------------------------
qp = QuadraticProgram("toy_insurance_qaoa_10")

n = 10
for i in range(n):
    qp.binary_var(name=f"x{i}")

# Toy "coverage values" (you can replace these with insurance scores, margins, etc.)
values = [8, 6, 5, 9, 7, 4, 10, 3, 6, 8]

# Maximize sum(values_i * x_i)
qp.maximize(linear={f"x{i}": values[i] for i in range(n)})

# Constraint: choose exactly 3 coverages
qp.linear_constraint(
    linear={f"x{i}": 1 for i in range(n)},
    sense="==",
    rhs=3,
    name="pick_3"
)

print("Quadratic Program:")
print(qp.prettyprint())


# -----------------------------
# 2. Set up QAOA
# -----------------------------
sampler = StatevectorSampler()
optimizer = COBYLA(maxiter=200)

# reps = p in QAOA language; larger reps = deeper circuit
qaoa = QAOA(
    sampler=sampler,
    optimizer=optimizer,
    reps=2
)

# Wrap QAOA so it can solve the QuadraticProgram
meo = MinimumEigenOptimizer(qaoa)


# -----------------------------
# 3. Solve
# -----------------------------
result = meo.solve(qp)

print("\nBest binary solution found:")
for i, bit in enumerate(result.x):
    print(f"x{i} = {int(bit)}")

print("\nObjective value:", result.fval)
print("Selected coverages:", [i for i, bit in enumerate(result.x) if round(bit) == 1])